# FIFA World Cup — Inference Notebook

Use this notebook to **predict match outcomes** using a model you already trained.

**You do NOT need to retrain here.** This notebook only loads saved files and runs predictions.

---

## Before you start

1. Run the **training notebook** (`fifa_world_cup_prediction.ipynb`) all the way through the **Save Files** cell.
2. Make sure these 6 files are in the same folder as this notebook:
   - `fifa_wc_model.keras`
   - `scaler.pkl`
   - `label_encoders.pkl`
   - `team_elo.pkl`
   - `team_history.pkl`
   - `model_metadata.pkl`

Run each cell below from top to bottom (Shift+Enter).

---
## Step 1: Import Libraries

**Inference** means using a trained model to make predictions on new data.

We only need a small set of libraries here — no training, no CSV loading.

In [ ]:
# Import os to check whether saved files exist on disk
import os

# Import pickle to load saved Python objects (.pkl files)
import pickle

# Import random for Monte Carlo tournament simulation
import random

# Import defaultdict to count tournament wins during simulation
from collections import defaultdict

# Import numpy for building feature arrays
import numpy as np

# Import pandas (optional but useful for displaying result tables)
import pandas as pd

# Import keras to load the saved neural network
from tensorflow import keras

# Confirm imports worked
print("Libraries imported successfully.")

---
## Step 2: Check That Saved Files Exist

Before loading, we verify all required files are present.
If anything is missing, go back to the training notebook and run the save cell.

In [ ]:
# List of files produced by the training notebook
REQUIRED_FILES = [
    "fifa_wc_model.keras",
    "scaler.pkl",
    "label_encoders.pkl",
    "team_elo.pkl",
    "team_history.pkl",
    "model_metadata.pkl",
]

# Loop through each required file and check if it exists
missing_files = []
for filename in REQUIRED_FILES:
    # os.path.isfile returns True if the file exists
    if os.path.isfile(filename):
        print(f"  OK  {filename}")
    else:
        print(f"MISSING  {filename}")
        missing_files.append(filename)

# Stop with a clear error if anything is missing
if missing_files:
    raise FileNotFoundError(
        "Missing saved model files: " + ", ".join(missing_files) +
        "\nRun the training notebook save cell first."
    )

print("\nAll required files found.")

---
## Step 3: Load the Saved Model and Helpers

We load:
- The **neural network** (`.keras` file)
- The **scaler** (how to normalize feature numbers)
- The **label encoders** (how text like "Brazil" was turned into numbers)
- **Elo ratings** and **match history** (for win-rate features)
- **Metadata** (feature column names and constants)

In [ ]:
# Load the trained Keras model from disk
model = keras.models.load_model("fifa_wc_model.keras")

# Open scaler.pkl in read-binary mode and load the StandardScaler object
with open("scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

# Load dictionary of LabelEncoders (one per categorical column)
with open("label_encoders.pkl", "rb") as f:
    label_encoders = pickle.load(f)

# Load final Elo rating for every team
with open("team_elo.pkl", "rb") as f:
    final_team_elo = pickle.load(f)

# Load match history lists used to compute recent win rates
with open("team_history.pkl", "rb") as f:
    final_team_history = pickle.load(f)

# Load metadata (feature names, default Elo, etc.)
with open("model_metadata.pkl", "rb") as f:
    metadata = pickle.load(f)

# Pull individual values out of the metadata dictionary
encoded_feature_columns = metadata["encoded_feature_columns"]
DEFAULT_ELO = metadata["DEFAULT_ELO"]

# Print confirmation and useful info
print("Model loaded successfully!")
print(f"Number of teams with Elo ratings: {len(final_team_elo)}")
print(f"Number of input features: {len(encoded_feature_columns)}")
print("Feature columns:", encoded_feature_columns)

---
## Step 4: Define Helper Functions

These small functions rebuild the same feature row the model saw during training.

They use the **saved** Elo, history, encoders, and scaler — not the raw CSV files.

In [ ]:
def get_tournament_weight(tournament_name):
    # Return a number showing how important a tournament type is
    if tournament_name is None or (isinstance(tournament_name, float) and np.isnan(tournament_name)):
        return 1
    name = str(tournament_name).lower()
    if name == "fifa world cup":
        return 5
    if "qualification" in name or "qualifier" in name:
        return 3
    if "friendly" in name:
        return 1
    return 2


def get_win_rate(team_name, history_dict):
    # Average result over the last 30 matches: 1=win, 0=loss, 0.5=draw
    last_30 = history_dict.get(team_name, [])[-30:]
    return float(np.mean(last_30)) if last_30 else 0.5


def encode_category(value, col_name):
    # Convert a text category to the integer the model expects
    le = label_encoders[col_name]
    value = str(value)
    if value in le.classes_:
        return float(le.transform([value])[0])
    # Unknown team/tournament seen at inference time -> "unknown" bucket
    return float(len(le.classes_))


def build_match_features(team_a, team_b, neutral=True, tournament="FIFA World Cup"):
    # Look up Elo (use DEFAULT_ELO if team name is unknown)
    home_elo = final_team_elo.get(team_a, DEFAULT_ELO)
    away_elo = final_team_elo.get(team_b, DEFAULT_ELO)

    # Build one feature dictionary for this match
    feature_row = {
        "home_team_elo": home_elo,
        "away_team_elo": away_elo,
        "elo_difference": home_elo - away_elo,
        "home_team_win_rate": get_win_rate(team_a, final_team_history),
        "away_team_win_rate": get_win_rate(team_b, final_team_history),
        "is_neutral_venue": 1 if neutral else 0,
        "tournament_weight": get_tournament_weight(tournament),
        "home_team_encoded": encode_category(team_a, "home_team"),
        "away_team_encoded": encode_category(team_b, "away_team"),
        "tournament_encoded": encode_category(tournament, "tournament"),
        "city_encoded": encode_category("Unknown", "city"),
        "country_encoded": encode_category("Unknown", "country"),
    }

    # Convert to a 2D numpy array with shape (1, num_features)
    X_one = np.array([[feature_row[col] for col in encoded_feature_columns]], dtype=float)
    # Apply the same scaling used during training
    return scaler.transform(X_one)


print("Helper functions ready.")

---
## Step 5: `predict_match` — Main Prediction Function

Call this with two team names to get win probabilities.

- `team_a` is treated as the **home** team in the feature vector
- For World Cup games, use `neutral=True` (default)
- Returns `(prob_team_a, prob_team_b)` as decimals between 0 and 1

In [ ]:
def predict_match(team_a, team_b, neutral=True, tournament="FIFA World Cup", verbose=True):
    # Build and scale features for this matchup
    X_scaled = build_match_features(team_a, team_b, neutral=neutral, tournament=tournament)

    # model.predict returns probability that team_a (home) wins
    prob_a = float(model.predict(X_scaled, verbose=0)[0][0])
    prob_b = 1.0 - prob_a

    if verbose:
        print(f"Match: {team_a} vs {team_b}")
        print(f"  Tournament: {tournament} | Neutral venue: {neutral}")
        print(f"  {team_a} win probability: {prob_a:.2%}")
        print(f"  {team_b} win probability: {prob_b:.2%}")
        # Simple pick: whichever probability is higher
        favorite = team_a if prob_a >= prob_b else team_b
        print(f"  Model favorite: {favorite}")

    return prob_a, prob_b


print("predict_match() is ready to use.")

---
## Step 6: Try Some Example Predictions

Edit the team names below to any teams you want.
Names must match the dataset style (e.g. `"United States"` not `"USA"`).

In [ ]:
# World Cup style: neutral venue, FIFA World Cup tournament weight
predict_match("Brazil", "Argentina")
print()
predict_match("France", "Germany")
print()
predict_match("United States", "Mexico")

---
## Step 7: Predict Multiple Matches at Once

Add as many matchups as you want to the list below.
Results are shown in a table.

In [ ]:
# List of (team_a, team_b) pairs to predict — edit this list freely
MATCHUPS = [
    ("Spain", "England"),
    ("Japan", "South Korea"),
    ("Morocco", "Senegal"),
    ("Argentina", "France"),
]

# Empty list to collect one row per match
rows = []

for team_a, team_b in MATCHUPS:
    # verbose=False so we don't print every match individually
    prob_a, prob_b = predict_match(team_a, team_b, verbose=False)
    rows.append({
        "Team A": team_a,
        "Team B": team_b,
        "A Win %": round(prob_a * 100, 1),
        "B Win %": round(prob_b * 100, 1),
        "Favorite": team_a if prob_a >= prob_b else team_b,
    })

# Convert list of dictionaries to a pandas DataFrame for nice display
results_table = pd.DataFrame(rows)
print(results_table.to_string(index=False))

---
## Step 8: View Top Teams by Elo Rating

**Elo** is a strength rating saved at the end of training.
Higher Elo = historically stronger team (based on all matches in the dataset).

In [ ]:
# Sort all teams by Elo from highest to lowest
sorted_elo = sorted(final_team_elo.items(), key=lambda x: x[1], reverse=True)

# Build a table of the top 15 teams
top_n = 15
elo_rows = [{"Rank": i + 1, "Team": team, "Elo": round(elo, 1)} for i, (team, elo) in enumerate(sorted_elo[:top_n])]
elo_table = pd.DataFrame(elo_rows)
print("Top 15 teams by Elo rating:")
print(elo_table.to_string(index=False))

---
## Step 9: Monte Carlo — Simulate 2026 World Cup (1,000 Runs)

This runs the full 48-team tournament **1,000 times** with randomness.
Each match uses the neural network's win probability.

**Note:** This cell can take several minutes because it simulates thousands of matches.

Edit `QUALIFIED_TEAMS_2026` if the official list changes.

In [ ]:
# 48 teams for 2026 FIFA World Cup — edit names to match the dataset
QUALIFIED_TEAMS_2026 = [
    "United States", "Canada", "Mexico",
    "Argentina", "Brazil", "Uruguay", "Colombia", "Ecuador", "Paraguay",
    "France", "Germany", "Spain", "England", "Portugal", "Netherlands", "Belgium",
    "Croatia", "Switzerland", "Austria", "Scotland", "Norway", "Denmark", "Poland", "Serbia",
    "Japan", "South Korea", "Australia", "Saudi Arabia", "Iran", "Qatar", "Jordan", "Uzbekistan",
    "Morocco", "Senegal", "Tunisia", "Algeria", "Egypt", "Ghana", "Cameroon", "Ivory Coast",
    "Costa Rica", "Panama", "Haiti", "New Zealand", "South Africa", "Curaçao", "Wales", "Cape Verde",
]

# Must be exactly 48 teams for the 2026 format used here
assert len(QUALIFIED_TEAMS_2026) == 48, "QUALIFIED_TEAMS_2026 must contain exactly 48 teams"


def simulate_one_match(team_a, team_b):
    # Get win probability without printing
    prob_a, _ = predict_match(team_a, team_b, neutral=True, tournament="FIFA World Cup", verbose=False)
    # Random draw: team_a wins if random number is below prob_a
    if random.random() < prob_a:
        return team_a
    return team_b


def play_group_stage(teams_in_group):
    # Track group-stage points (3 for a win)
    points = {team: 0 for team in teams_in_group}
    # Every pair in the group plays once
    for i in range(len(teams_in_group)):
        for j in range(i + 1, len(teams_in_group)):
            winner = simulate_one_match(teams_in_group[i], teams_in_group[j])
            points[winner] += 3
    # Return teams sorted by points (highest first)
    return sorted(points.items(), key=lambda x: x[1], reverse=True)


def simulate_knockout(teams):
    remaining = list(teams)
    random.shuffle(remaining)
    while len(remaining) > 1:
        next_round = []
        for i in range(0, len(remaining), 2):
            if i + 1 >= len(remaining):
                next_round.append(remaining[i])
            else:
                next_round.append(simulate_one_match(remaining[i], remaining[i + 1]))
        remaining = next_round
    return remaining[0]


def simulate_one_world_cup():
    teams = QUALIFIED_TEAMS_2026.copy()
    random.shuffle(teams)
    groups = [teams[i * 4:(i + 1) * 4] for i in range(12)]

    group_winners, group_runners_up, third_place = [], [], []
    for group in groups:
        ranking = play_group_stage(group)
        group_winners.append(ranking[0][0])
        group_runners_up.append(ranking[1][0])
        third_place.append(ranking[2][0])

    knockout_teams = group_winners + group_runners_up
    third_place_sorted = sorted(third_place, key=lambda t: final_team_elo.get(t, DEFAULT_ELO), reverse=True)
    knockout_teams += third_place_sorted[:8]

    return simulate_knockout(knockout_teams)


# Number of full tournament simulations
NUM_SIMULATIONS = 1000
win_counts = defaultdict(int)

print(f"Running {NUM_SIMULATIONS} Monte Carlo simulations... (this may take a few minutes)")
for sim in range(NUM_SIMULATIONS):
    champion = simulate_one_world_cup()
    win_counts[champion] += 1
    # Print progress every 100 simulations
    if (sim + 1) % 100 == 0:
        print(f"  Completed {sim + 1}/{NUM_SIMULATIONS}...")

# Convert win counts to probabilities
win_probs = {team: count / NUM_SIMULATIONS for team, count in win_counts.items()}
top_teams = sorted(win_probs.items(), key=lambda x: x[1], reverse=True)

print("\nTop 10 most likely 2026 World Cup winners:")
for rank, (team, prob) in enumerate(top_teams[:10], start=1):
    print(f"{rank:2d}. {team:<20} {prob:.2%}")

---
## Step 10: Look Up a Single Team

Quick helper to see a team's Elo and recent win rate before predicting a match.

In [ ]:
def show_team_info(team_name):
    # Get Elo or default if team is unknown
    elo = final_team_elo.get(team_name, DEFAULT_ELO)
    # Get recent win rate from saved history
    win_rate = get_win_rate(team_name, final_team_history)
    # Count how many matches are in history for this team
    num_matches = len(final_team_history.get(team_name, []))

    print(f"Team: {team_name}")
    print(f"  Elo rating: {elo:.1f}")
    print(f"  Recent win rate (last 30): {win_rate:.2%}")
    print(f"  Total matches in history: {num_matches}")
    if team_name not in final_team_elo:
        print("  Warning: team not found in saved Elo data — using default Elo 1500.")


# Change the team name below to any team you are curious about
show_team_info("Brazil")
print()
show_team_info("United States")